# Notebook 02 — Data Cleaning
**Bluestock MF Capstone**

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('..')
RAW  = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'
PROC.mkdir(exist_ok=True)


## Clean NAV History — parse dates, ffill weekends, dedup

In [ ]:

nav = pd.read_csv(RAW / '02_nav_history.csv', low_memory=False)
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')
nav['nav']  = pd.to_numeric(nav['nav'], errors='coerce')
nav = nav.dropna(subset=['date','nav'])
nav = nav[nav['nav'] > 0]
nav = nav.sort_values(['amfi_code','date']).drop_duplicates(subset=['amfi_code','date'])

# Forward-fill weekends / holidays
all_dates = pd.date_range(nav['date'].min(), nav['date'].max(), freq='D')
chunks = []
for code in nav['amfi_code'].unique():
    sub = nav[nav['amfi_code']==code].set_index('date')['nav']
    sub = sub.reindex(all_dates).ffill().reset_index()
    sub.columns = ['date','nav']
    sub['amfi_code'] = code
    chunks.append(sub)
nav_clean = pd.concat(chunks, ignore_index=True)[['amfi_code','date','nav']]
nav_clean = nav_clean.dropna(subset=['nav'])
print(f'nav_history: {len(nav):,} raw → {len(nav_clean):,} clean (after ffill)')
nav_clean.to_csv(PROC / '02_nav_history_clean.csv', index=False)


## Clean Investor Transactions

In [ ]:

txns = pd.read_csv(RAW / '08_investor_transactions.csv')
txns['transaction_date'] = pd.to_datetime(txns['transaction_date'], errors='coerce')
txns['amount_inr'] = pd.to_numeric(txns['amount_inr'], errors='coerce')
txns = txns[txns['amount_inr'] > 0]
txns['transaction_type'] = txns['transaction_type'].str.strip().str.title()
txns['kyc_status'] = txns['kyc_status'].str.strip().str.title()
txns = txns.drop_duplicates()
print('Type distribution:', txns['transaction_type'].value_counts().to_dict())
print('KYC status:', txns['kyc_status'].value_counts().to_dict())
txns.to_csv(PROC / '08_investor_txns_clean.csv', index=False)
print(f'investor_txns cleaned: {txns.shape}')


## Clean Scheme Performance — validate numeric, flag anomalies

In [ ]:

sp = pd.read_csv(RAW / '07_scheme_performance.csv')
num_cols = ['return_1yr_pct','return_3yr_pct','return_5yr_pct','alpha','beta',
            'sharpe_ratio','sortino_ratio','std_dev_ann_pct','max_drawdown_pct',
            'aum_crore','expense_ratio_pct']
for c in num_cols:
    sp[c] = pd.to_numeric(sp[c], errors='coerce')
anomalies = sp[sp['expense_ratio_pct'] > 2.5]
print(f'Expense ratio anomalies (>2.5%): {len(anomalies)}')
sp.to_csv(PROC / '07_scheme_performance_clean.csv', index=False)
print(f'scheme_performance cleaned: {sp.shape}')


## Summary

In [ ]:

import os
files = list(Path(PROC).glob('*.csv'))
print(f'Processed files: {len(files)}')
for f in sorted(files):
    df = pd.read_csv(f, nrows=1)
    print(f'  {f.name}')
